### Instalações

In [ ]:
# Instalar pacotes necessários (se ainda não instalados)
!pip install nltk scikit-learn

!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.0 MB/s eta 0:00:00


### Importações

In [ ]:
import glob
import json
import pandas as pd
import re
import unidecode
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

In [ ]:
# prompt: conect ao drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Tratamento dos dados

In [ ]:
# Configurar NLTK (stopwords)

nltk.download('stopwords') # Baixar recursos do NLTK
stopwords_pt = set(stopwords.words('portuguese')) # Configurações

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Pré processamento (removendo palavras chaves)
Remoção de:
- stopwords
- #
- @  


In [ ]:
# Conjunto de stopwords da língua portuguesa
stopwords_pt = set([
    # Adicione aqui suas stopwords
    'de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'é', 'com', 'não', 'uma', 'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'foi', 'ao', 'ele', 'das', 'tem', 'à', 'seu', 'sua', 'ou', 'ser', 'quando', 'muito', 'há', 'nos', 'já', 'está', 'eu', 'também', 'só', 'pelo', 'pela', 'até', 'isso', 'ela', 'entre', 'era', 'depois', 'sem', 'mesmo', 'aos', 'ter', 'seus', 'quem', 'nas', 'me', 'esse', 'eles', 'estão', 'você', 'tinha', 'foram', 'essa', 'num', 'nem', 'suas', 'meu', 'às', 'minha', 'têm', 'numa', 'pelos', 'elas', 'havia', 'seja', 'qual', 'será', 'nós', 'tenho', 'lhe', 'deles', 'essas', 'esses', 'pelas', 'este', 'dele', 'tu', 'te', 'vocês', 'vos', 'lhes', 'meus', 'minhas', 'teu', 'tua', 'teus', 'tuas', 'nosso', 'nossa', 'nossos', 'nossas', 'dela', 'delas', 'esta', 'estes', 'estas', 'aquele', 'aquela', 'aqueles', 'aquelas', 'isto', 'aquilo', 'estou', 'está', 'estamos', 'estão', 'estive', 'esteve', 'estivemos', 'estiveram', 'estava', 'estávamos', 'estavam', 'estivera', 'estivéramos', 'esteja', 'estejamos', 'estejam', 'estivesse', 'estivéssemos', 'estivessem', 'estiver', 'estivermos', 'estiverem', 'hei', 'há', 'havemos', 'hão', 'houve', 'houvemos', 'houveram', 'houvera', 'houvéramos', 'haja', 'hajamos', 'hajam', 'houvesse', 'houvéssemos', 'houvessem', 'houver', 'houvermos', 'houverem', 'houverei', 'houverá', 'houveremos', 'houverão', 'houveria', 'houveríamos', 'houveriam', 'sou', 'somos', 'são', 'era', 'éramos', 'eram', 'fui', 'foi', 'fomos', 'foram', 'fora', 'fôramos', 'seja', 'sejamos', 'sejam', 'fosse', 'fôssemos', 'fossem', 'for', 'formos', 'forem', 'serei', 'será', 'seremos', 'serão', 'seria', 'seríamos', 'seriam', 'tenho', 'tem', 'temos', 'têm', 'tinha', 'tínhamos', 'tinham', 'tive', 'teve', 'tivemos', 'tiveram', 'tivera', 'tivéramos', 'tenha', 'tenhamos', 'tenham', 'tivesse', 'tivéssemos', 'tivessem', 'tiver', 'tivermos', 'tiverem', 'terei', 'terá', 'teremos', 'terão', 'teria', 'teríamos', 'teriam'
])

# Palavras a serem excluídas além das stopwords
palavras_excluir = {
    'aracaju', 'emilia', 'correa', 'luiz', 'roberto',
    'fortaleza', 'andre', 'fernandes', 'evandro', 'leitao',
    'joao', 'pessoa', 'cicero', 'lucena', 'luciano', 'cartaxo',
    'maceio', 'jhc', 'rafael', 'brito',
    'natal', 'natalia', 'bonavides', 'paulinho', 'freire',
    'recife', 'gilson', 'machado', 'neto', 'campos',
    'salvador', 'bruno', 'reis', 'geraldo', 'jr', 'kleber', 'rosa',
    'sao', 'luis', 'duarte', 'eduardo', 'braide',
    'teresina', 'fabio', 'novo', 'silvio', 'mendes', 'marcelo', 'queiroga', 'propaganda', 'eleitoral'
}

# Função de pré-processamento de texto
def preprocessar_texto(texto):
    if pd.isna(texto) or texto.strip() == '':
        return ''

    texto = texto.lower()
    texto = re.sub(r"http\S+|www\S+", '', texto)  # Remover URLs
    texto = re.sub(r'#\S+|@\S+', '', texto)       # Remover hashtags e menções
    texto = re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', texto)  # Remover caracteres não alfabéticos
    texto = unidecode.unidecode(texto)            # Remover acentos

    palavras = texto.split()
    palavras_filtradas = [p for p in palavras if p not in stopwords_pt and p not in palavras_excluir]

    return ' '.join(palavras_filtradas)


In [ ]:
# Caminho para os arquivos JSON
caminho_json = "/content/drive/MyDrive/Colab Notebooks/ENIAC/2.0/politicos/*.json"
arquivos = glob.glob(caminho_json)

politicos = []
textos_por_politico = []

# Ler arquivos e processar textos
for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)
        texto_completo = ' '.join(df['texto_processado'].dropna().tolist())
        textos_por_politico.append(texto_completo)
        nome_politico = os.path.basename(arquivo).replace('.json', '')
        politicos.append(nome_politico)

In [ ]:
# Vetorização TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(textos_por_politico)
termos = vectorizer.get_feature_names_out()

# DataFrame com soma dos TF-IDF por termo para cada político
df_tfidf_politicos = pd.DataFrame(X_tfidf.toarray(), columns=termos, index=politicos)

### Wordclouds

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Gerar uma WordCloud para cada político
for idx, politico in enumerate(politicos):
    vetor = X_tfidf[idx].toarray().flatten()
    pesos_termos = dict(zip(termos, vetor))

    wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2')
    wordcloud.generate_from_frequencies(pesos_termos)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"WordCloud - {politico}", fontsize=18)
    # Substituir espaços ou caracteres inválidos no nome do arquivo
    nome_arquivo = f"wordclouds/{politico.replace(' ', '_').replace('/', '-')}.png"
    plt.savefig(nome_arquivo, bbox_inches='tight')
    plt.close()  # Fecha a figura para economizar memória

In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_por_politico.txt", "w", encoding="utf-8") as f:
    for idx, politico in enumerate(politicos):
        vetor = X_tfidf[idx].toarray().flatten()
        pesos_termos = dict(zip(termos, vetor))
        top_10 = sorted(pesos_termos.items(), key=lambda x: x[1], reverse=True)[:10]

        f.write(f"\nTop 10 termos de {politico}:\n\n")
        for termo, peso in top_10:
            f.write(f"{termo} ({peso:.3f})\n")

In [ ]:
# prompt: Imprima os 150 termos mais frequentes por politico

# Imprimir os 150 termos mais frequentes usando heap queue
import heapq as hq

with open("top_150_termos_por_politico.txt", "w", encoding="utf-8") as f:
    for idx, politico in enumerate(politicos):
        vetor = X_tfidf[idx].toarray().flatten()
        pesos_termos = dict(zip(termos, vetor))
        # Usar nlargest para encontrar os 50 termos com maiores pesos
        top_150 = hq.nlargest(150, pesos_termos.items(), key=lambda item: item[1])

        f.write(f"\n🔹 Top 150 termos de {politico}:\n\n")
        for termo, peso in top_150:
            f.write(f"{termo} ({peso:.3f})\n")

### Base de Dados

In [ ]:
# Criar DataFrame com os valores TF-IDF
df_tfidf_politicos = pd.DataFrame(X_tfidf.toarray(), index=politicos, columns=termos)

# === GERAR BASE BINÁRIA ===
mediana_global = df_tfidf_politicos.median()
df_binaria = df_tfidf_politicos.gt(mediana_global).astype(int)
df_binaria.reset_index(inplace=True)
df_binaria.rename(columns={'index': 'politico_nome'}, inplace=True)

In [ ]:
# prompt: adicione duas colunas, orientação e vitoria, ao dataframe df_binaria para que eu possa preencher com um dicionario de dados

df_binaria['orientacao'] = ''
df_binaria['vitoria'] = ''

In [ ]:
# Reorganiza as colunas: 'politico', 'orientação', 'vitoria', ...
cols = ['politico_nome', 'orientacao', 'vitoria'] + [col for col in df_binaria.columns if col not in ['politico_nome', 'orientacao', 'vitoria']]
df_binaria = df_binaria[cols]

In [ ]:
df_binaria

,politico_nome,orientacao,vitoria,abandono,aberto,abraco,acabar,acao,acessibilidade,acessivel,...,vota,votar,vote,voto,votos,vou,voz,youtube,zero,zona
0,sao_luis_eduardo_braide,,,0,0,1,1,1,0,0,...,0,1,0,0,0,1,0,0,0,1
1,aracaju_emilia_correa,,,1,1,0,0,1,1,0,...,0,0,0,0,1,0,0,0,0,1
2,natal_natalia_bonavides,,,1,1,0,1,0,0,0,...,0,1,0,1,1,0,1,1,1,1
3,salvador_geraldo_junior,,,0,0,1,0,0,0,1,...,1,1,1,0,0,0,0,1,0,0
4,teresina_fabio_novo,,,0,0,0,0,1,1,1,...,0,0,1,1,0,0,1,1,1,1
5,teresina_silvio_mendes,,,1,1,1,0,0,0,1,...,1,1,1,1,1,0,1,1,1,1
6,recife_jc,,,0,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,1,1
7,fortaleza_evandro_leitao,,,0,0,0,1,0,1,0,...,1,0,1,1,1,1,1,1,0,0
8,salvador_bruno_reis,,,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
9,fortaleza_andre_fernandes,,,0,0,0,1,0,0,0,...,1,0,1,0,0,0,0,0,0,0


In [ ]:
# Dicionário com os dados fornecidos
dados_politicos = {
    'aracaju_luiz_roberto':       ('esq', 'não'),
    'maceio_rafael_brito':        ('cen', 'não'),
    'sao_luis_eduardo_braide':    ('cen', 'sim'),
    'salvador_bruno_reis':        ('dir', 'sim'),
    'maceio_jhc':                 ('dir', 'sim'),
    'sao_luis_duarte_jr':         ('esq', 'não'),
    'joao_pessoa_cicero_lucena':  ('dir', 'sim'),
    'natal_natalia_bonavides':    ('esq', 'não'),
    'salvador_kleber_rosa':       ('esq', 'não'),
    'salvador_geraldo_junior':    ('cen', 'não'),
    'aracaju_emilia_correa':      ('dir', 'sim'),
    'teresina_silvio_mendes':     ('dir', 'sim'),
    'recife_jc':                  ('esq', 'sim'),
    'natal_paulinho_freire':      ('dir', 'sim'),
    'teresina_fabio_novo':        ('esq', 'não'),
    'fortaleza_andre_fernandes':  ('dir', 'não'),
    'recife_gmn':                 ('dir', 'não'),
    'joao_pessoa_luciano_cartaxo':('esq', 'não'),
    'fortaleza_evandro_leitao':   ('esq', 'sim'),
    'joao_pessoa_marcelo_queiroga': ('dir', 'não')
}

# Preenche as colunas com base no dicionário
for index, row in df_binaria.iterrows():
    chave = row['politico_nome']
    if chave in dados_politicos:
        df_binaria.at[index, 'orientacao'] = dados_politicos[chave][0]
        df_binaria.at[index, 'vitoria'] = dados_politicos[chave][1]

In [ ]:
# Salvar no CSV
df_binaria.to_csv("base_de_dados.csv", index=False)

In [ ]:
# prompt: faça duas colunas. A primeira são todas as palavras do vocabulário em ordem alfabética e a segunda a frequências com que cada uma delas apareceu considerando todos as postagens

from collections import Counter

# Concatenar todos os textos processados em uma única string
texto_total = ' '.join(textos_por_politico)

# Dividir a string em palavras
palavras_total = texto_total.split()

# Contar a frequência de cada palavra
contagem_palavras = Counter(palavras_total)

# Ordenar as palavras alfabeticamente
palavras_ordenadas = sorted(contagem_palavras.keys())

# Criar listas para o DataFrame
coluna_palavra = palavras_ordenadas
coluna_frequencia = [contagem_palavras[palavra] for palavra in palavras_ordenadas]

# Criar o DataFrame
df_vocabulario_frequencia = pd.DataFrame({
    'Palavra': coluna_palavra,
    'Frequencia': coluna_frequencia
})

# Imprimir o DataFrame
df_vocabulario_frequencia
df_vocabulario_frequencia.to_csv("frequencia_do_vocabulario.csv", index=False)